In [ ]:
import os

import cmocean
import lmdb
import numpy as np
from matplotlib import pyplot as plt

# Visualize ESDC LST samples stored in LMDB.
# Use this to verify that LMDB creation preserves the samples.
# Each sample: (T=5, C=2, H=64, W=64), channel 0 = LST.
# NaNs indicate gaps. The middle time step (t=2) is the reconstruction target.

_esdc_data_dir = os.environ.get(
    "ESDC_DATA_DIR",
    "../../../esdc/data/lst_no_syn_gaps",
)
lmdb_path = os.path.join(_esdc_data_dir, "lmdb", "dataset.lmdb")

n_samples = 10
n_timesteps = 5
seed = None  # set to an int for reproducibility
TARGET_T = n_timesteps // 2
VAL_RANGE = (180, 360)
CMAP = cmocean.cm.matter_r
VMIN, VMAX = 273.15, 323.15

# Close a previously opened environment (important when re-running the cell).
try:
    if "env" in globals() and env is not None:
        env.close()
except Exception:
    pass

env = lmdb.open(
    lmdb_path,
    readonly=True,
    lock=False,
    readahead=False,
    meminit=False,
    subdir=False,
)

n_total = env.stat()["entries"]
if n_total == 0:
    raise RuntimeError(f"LMDB is empty: {lmdb_path}")


def _decode(data: bytes) -> np.ndarray:
    ndim = int(np.frombuffer(data, dtype=np.int32, count=1, offset=0)[0])
    shape = tuple(np.frombuffer(data, dtype=np.int32, count=ndim, offset=4).tolist())
    header_nbytes = 4 * (1 + ndim)
    return np.frombuffer(data, dtype=np.float32, offset=header_nbytes).reshape(shape)


def load_sample(idx: int) -> np.ndarray:
    with env.begin(write=False) as txn:
        data = txn.get(str(idx).encode())
    if data is None:
        raise IndexError(f"No entry for index {idx}")
    return _decode(data)


# Reservoir-sample n_samples indices from [0, n_total)
rng = np.random.default_rng(seed)
chosen_indices = []
for i in range(n_total):
    if len(chosen_indices) < n_samples:
        chosen_indices.append(i)
    else:
        j = int(rng.integers(0, i + 1))
        if j < n_samples:
            chosen_indices[j] = i
chosen_indices = sorted(chosen_indices)

fig, axes = plt.subplots(
    len(chosen_indices), n_timesteps, figsize=(20, 4 * len(chosen_indices))
)
if len(chosen_indices) == 1:
    axes = [axes]

for i, idx in enumerate(chosen_indices):
    sample = load_sample(idx)  # (T, C, H, W)
    lst = sample[:, 0, :, :].astype(np.float32)  # (T, H, W), LST channel
    lst = lst * (VAL_RANGE[1] - VAL_RANGE[0]) + VAL_RANGE[0]  # → Kelvin

    for t in range(n_timesteps):
        ax = axes[i][t]
        ax.matshow(lst[t], cmap=CMAP, vmin=VMIN, vmax=VMAX)

        H, W = lst[t].shape
        frac = 100.0 * float(np.isnan(lst[t]).sum()) / float(lst[t].size)
        title = f"Sample {i + 1} (idx={idx}), t={t}\n{H}\u00d7{W}  gaps={frac:.1f}%"
        if t == TARGET_T:
            title += " (target)"
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()